In [1]:
# Imports
import torch
import torch.nn as nn

In [3]:
#1. LoRA adaptation layer
class LoRALinear(nn.Module):
    def __init__(self, linear_layer, rank=4, alpha=1):
        """
        LoRA adaptation for a Linear layer

        Args:
            linear_layer: Pre-trained nn.Linear layer to adapt
            rank: Rank of the low-rank decomposition (r)
            alpha: Scaling factor for LoRA weights
        """
        super().__init__()

        self.in_features = linear_layer.in_features
        self.out_features = linear_layer.out_features
        self.rank = rank
        self.alpha = alpha

        # Freeze original weights
        self.linear = linear_layer
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False

        # LoRA matrices: W + BA (where B is out_features x rank, A is rank x in_features)
        # A initialized with random Gaussian, B initialized with zeros
        self.lora_A = nn.Parameter(torch.randn(rank, self.in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank))

        # Scaling factor
        self.scaling = self.alpha / self.rank

    def forward(self, x):
        # Original linear transformation (frozen)
        original_output = self.linear(x)

        # LoRA adaptation: (B @ A) @ x, scaled by alpha/rank
        lora_output = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling

        return original_output + lora_output



In [5]:
# 2. Demonstrations

# a). Create a standard linear layer
original_linear = nn.Linear(1024, 1024)
print("Original Linear Layer:")
print(f"  Total parameters: {sum(p.numel() for p in original_linear.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in original_linear.parameters() if p.requires_grad):,}\n")

# b). Wrap it with LoRA (rank=8)
lora_linear = LoRALinear(original_linear, rank=8, alpha=16)
print("LoRA-adapted Linear Layer (rank=8):")
print(f"  Total parameters: {sum(p.numel() for p in lora_linear.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in lora_linear.parameters() if p.requires_grad):,}")

# c). Calculate efficiency gain
total_params = sum(p.numel() for p in lora_linear.parameters())
trainable_params = sum(p.numel() for p in lora_linear.parameters() if p.requires_grad)
print(f" Trainable: {100 * trainable_params / total_params:.2f}% of total")
print(f" Reduction: {total_params / trainable_params:.1f}x fewer trainable parameters")

# d). Test forward pass
x = torch.randn(4, 1024)
output = lora_linear(x)
print(f" Forward pass successful! Output shape: {output.shape}")


Original Linear Layer:
  Total parameters: 1,049,600
  Trainable parameters: 1,049,600

LoRA-adapted Linear Layer (rank=8):
  Total parameters: 1,065,984
  Trainable parameters: 16,384
 Trainable: 1.54% of total
 Reduction: 65.1x fewer trainable parameters
 Forward pass successful! Output shape: torch.Size([4, 1024])
